In [1]:
import os

In [2]:
%pwd

'd:\\End_to_End_ML_project_for_Gems_Price_Prediction\\research'

In [3]:
os.chdir("../")

In [4]:
%pwd

'd:\\End_to_End_ML_project_for_Gems_Price_Prediction'

In [5]:
from dataclasses import dataclass
from pathlib import Path


@dataclass(frozen=True)
class DataTransformationConfig:
    root_dir: Path
    data_path: Path

In [6]:
from mlProject.constants import *
from mlProject.utils.common import read_yaml, create_directories

In [7]:
class ConfigurationManager:
    def __init__(self, config_filepath = CONFIG_FILE_PATH, params_filepath = PARAMS_FILE_PATH, schema_filepath = SCHEMA_FILE_PATH):

        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        self.schema = read_yaml(schema_filepath)

        create_directories([self.config.artifacts_root])


    
    def get_data_transformation_config(self) -> DataTransformationConfig:
        config = self.config.data_transformation

        create_directories([config.root_dir])

        data_transformation_config = DataTransformationConfig(
            root_dir=config.root_dir,
            data_path=config.data_path,
        )

        return data_transformation_config

In [9]:
import os
from mlProject import logger
from sklearn.preprocessing import StandardScaler 
from sklearn.model_selection import train_test_split
import pandas as pd

In [10]:
import os
from mlProject import logger
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
import pandas as pd
import numpy as np

class DataTransformation:
    def __init__(self, config: DataTransformationConfig):
        self.config = config


    def preprocessing(self):
        """
        Main preprocessing function that handles:
        - Loading data
        - Dropping irrelevant columns
        - Handling missing values
        - Encoding categorical variables
        - Scaling numerical features
        - Splitting into train/test sets
        """
        data = pd.read_csv(self.config.data_path)
        logger.info(f"Original data shape: {data.shape}")

        # Drop id column as it is statistically insignificant
        data = data.drop(labels=['id'], axis=1)
        logger.info("Dropped 'id' column")

        # Identify numerical and categorical columns
        numerical_columns = list(data.columns[data.dtypes != 'object'])
        categorical_columns = list(data.columns[data.dtypes == 'object'])

        logger.info(f"Numerical columns: {numerical_columns}")
        logger.info(f"Categorical columns: {categorical_columns}")

        # Handle missing values
        data = self._handle_missing_values(data, numerical_columns, categorical_columns)

        # Encode categorical variables
        data = self._encode_categorical(data, categorical_columns)

        # Scale numerical features
        data = self._scale_numerical(data, numerical_columns)

        # Split the data into training and test sets (0.80, 0.20 split)
        train, test = train_test_split(data, test_size=0.20, random_state=42)

        # Save preprocessed data
        train.to_csv(os.path.join(self.config.root_dir, "train.csv"), index=False)
        test.to_csv(os.path.join(self.config.root_dir, "test.csv"), index=False)

        logger.info("Preprocessed data split into training and test sets")
        logger.info(f"Training set shape: {train.shape}")
        logger.info(f"Test set shape: {test.shape}")

        print(f"Training set shape: {train.shape}")
        print(f"Test set shape: {test.shape}")

        return train, test


    def _handle_missing_values(self, data, numerical_columns, categorical_columns):
        """Handle missing values in the dataset"""
        logger.info("Handling missing values...")
        
        # For numerical columns, fill with median
        for col in numerical_columns:
            if data[col].isnull().sum() > 0:
                data[col].fillna(data[col].median(), inplace=True)
                logger.info(f"Filled missing values in {col} with median")
        
        # For categorical columns, fill with mode
        for col in categorical_columns:
            if data[col].isnull().sum() > 0:
                data[col].fillna(data[col].mode()[0], inplace=True)
                logger.info(f"Filled missing values in {col} with mode")
        
        return data


    def _encode_categorical(self, data, categorical_columns):
        """Encode categorical variables using LabelEncoder"""
        logger.info("Encoding categorical variables...")
        
        for col in categorical_columns:
            le = LabelEncoder()
            data[col] = le.fit_transform(data[col])
            logger.info(f"Encoded {col}")
        
        return data


    def _scale_numerical(self, data, numerical_columns):
        """Scale numerical features using StandardScaler"""
        logger.info("Scaling numerical features...")
        
        scaler = StandardScaler()
        data[numerical_columns] = scaler.fit_transform(data[numerical_columns])
        logger.info(f"Scaled {len(numerical_columns)} numerical columns")
        
        return data


In [11]:
try:
    config = ConfigurationManager()
    data_transformation_config = config.get_data_transformation_config()
    data_transformation = DataTransformation(config=data_transformation_config)
    train, test = data_transformation.preprocessing()
except Exception as e:
    raise e


[2026-02-05 20:20:37,774: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-02-05 20:20:37,797: INFO: common: yaml file: params.yaml loaded successfully]
[2026-02-05 20:20:37,821: INFO: common: yaml file: schema.yaml loaded successfully]
[2026-02-05 20:20:37,821: INFO: common: created directory at: artifacts]
[2026-02-05 20:20:37,821: INFO: common: created directory at: artifacts/data_transformation]


[2026-02-05 20:20:38,142: INFO: 4190402376: Original data shape: (193573, 11)]
[2026-02-05 20:20:38,179: INFO: 4190402376: Dropped 'id' column]
[2026-02-05 20:20:38,185: INFO: 4190402376: Numerical columns: ['carat', 'depth', 'table', 'x', 'y', 'z', 'price']]
[2026-02-05 20:20:38,185: INFO: 4190402376: Categorical columns: ['cut', 'color', 'clarity']]
[2026-02-05 20:20:38,185: INFO: 4190402376: Handling missing values...]
[2026-02-05 20:20:38,240: INFO: 4190402376: Encoding categorical variables...]
[2026-02-05 20:20:38,281: INFO: 4190402376: Encoded cut]
[2026-02-05 20:20:38,348: INFO: 4190402376: Encoded color]
[2026-02-05 20:20:38,396: INFO: 4190402376: Encoded clarity]
[2026-02-05 20:20:38,396: INFO: 4190402376: Scaling numerical features...]
[2026-02-05 20:20:38,448: INFO: 4190402376: Scaled 7 numerical columns]
[2026-02-05 20:20:42,609: INFO: 4190402376: Preprocessed data split into training and test sets]
[2026-02-05 20:20:42,611: INFO: 4190402376: Training set shape: (154858, 1